# 76 - Experimento hibrido TF-IDF + MPNet + HDBSCAN

Este notebook combina a representacao hibrida do notebook 52 com a avaliacao e inspecao dos notebooks 74 e 75. Cada descricao recebe um vetor TF-IDF normalizado e um embedding MPNet normalizado; os blocos sao ponderados por `ALPHA` e concatenados antes do agrupamento.

**Pergunta:** a combinacao de sinais lexicais e semanticos forma grupos densos uteis sob diferentes configuracoes TF-IDF e parametros HDBSCAN?

A analise usa descricoes normalizadas unicas, sem NCM ou rotulos humanos, e contabiliza ruido explicitamente. A grade cruza cinco analyzers TF-IDF com nove configuracoes HDBSCAN (45 ajustes). O custo pode ser significativo.


## 1. Configuracoes

Os parametros de representacao seguem o notebook 52. A grade HDBSCAN, a avaliacao e a amostragem para visualizacao seguem os notebooks 74 e 75. Cada execucao cria uma pasta nova em `reports/tfidf_mpnet_hdbscan/`.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook76")
pd.set_option("display.max_colwidth", 140)

DATASET_NAME = "2025_NCMs_filtrados-sem-duplicatas"
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(f"Nome de arquivo inesperado: {DATASET_PATH.name}")
PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]

# Parametros de representacao do notebook 52.
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = 10_000  # None usa todas as descricoes.
SAMPLE_RANDOM_STATE = 42
ENCODE_BATCH_SIZE = 64
ANALYZER_CONFIGS = {
    "char_wb_3_5": ("char_wb", (3, 5)),
    "char_wb_5_10": ("char_wb", (5, 10)),
    "word_1_3": ("word", (1, 3)),
    "word_1_5": ("word", (1, 5)),
    "word_2_5": ("word", (2, 5)),
}
TFIDF_MIN_DF = 5
TFIDF_MAX_FEATURES = 30_000
ALPHA = 0.2

# Grade HDBSCAN e avaliacao dos notebooks 74 e 75.
MIN_CLUSTER_SIZE_VALUES = [5, 10, 20]
MIN_SAMPLES_VALUES = [None, 5, 10]
CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_N_JOBS = 1
SILHOUETTE_SAMPLE_SIZE = 10_000
MAX_CLUSTERS_FOR_RANKING = 200
SILHOUETTE_WEIGHT = 0.7
NOISE_WEIGHT = 0.5
TSNE_MAX_POINTS = 5_000
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_SVD_COMPONENTS = 50
TSNE_3D_N_COMPONENTS = 3
RANKING_FILTER_SILHOUETTE_ABOVE_NOISE = True
SHOW_FULL_BASE_IF_RANKING_EMPTY = True  # Se os filtros eliminarem tudo, usa todas as configuracoes.
SELECTED_RUN_ID = None  # None seleciona o primeiro run do ranking filtrado.
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "tfidf_mpnet_hdbscan"
MODEL_ROOT = PROJECT_ROOT / "models" / "tfidf_mpnet_hdbscan"


## 2. Funcoes da analise

As funcoes mantem no notebook a preparacao, a combinacao das representacoes, a avaliacao e a inspecao dos clusters. A concatenacao preserva TF-IDF esparso e valida a ordem das linhas.


In [2]:
def prepare_items(frame):
    """Limpa descricoes, agrega duplicatas e preserva a frequencia total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY].sum()
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A preparacao alterou a soma de frequencia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)


def combine_representations(tfidf_matrix, mpnet_matrix, alpha, tfidf_ids, mpnet_ids):
    """Normaliza e concatena TF-IDF e MPNet com pesos controlados por alpha."""
    try:
        alpha = float(alpha)
    except (TypeError, ValueError) as exc:
        raise ValueError("alpha deve ser um numero finito no intervalo [0, 1].") from exc
    if not np.isfinite(alpha) or not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha deve ser um numero finito no intervalo [0, 1].")
    if tfidf_matrix.ndim != 2 or mpnet_matrix.ndim != 2:
        raise ValueError("As representacoes devem ser matrizes bidimensionais.")
    if tfidf_matrix.shape[0] != mpnet_matrix.shape[0]:
        raise ValueError("As representacoes devem ter a mesma quantidade de linhas.")
    tfidf_ids, mpnet_ids = np.asarray(tfidf_ids), np.asarray(mpnet_ids)
    rows = tfidf_matrix.shape[0]
    if len(tfidf_ids) != rows or len(mpnet_ids) != rows:
        raise ValueError("Cada matriz deve ter um identificador por linha.")
    if len(np.unique(tfidf_ids)) != rows or len(np.unique(mpnet_ids)) != rows:
        raise ValueError("Os identificadores devem ser unicos em cada representacao.")
    if not np.array_equal(tfidf_ids, mpnet_ids):
        raise ValueError("A ordem dos registros difere entre TF-IDF e MPNet.")
    if alpha == 1.0:
        return normalize(tfidf_matrix, norm="l2", axis=1, copy=True)
    if alpha == 0.0:
        return normalize(mpnet_matrix, norm="l2", axis=1, copy=True)
    tfidf_normalized = normalize(tfidf_matrix, norm="l2", axis=1, copy=True)
    mpnet_normalized = normalize(mpnet_matrix, norm="l2", axis=1, copy=True)
    tfidf_weighted = tfidf_normalized * np.sqrt(alpha)
    mpnet_weighted = mpnet_normalized * np.sqrt(1.0 - alpha)
    if sparse.issparse(tfidf_weighted):
        return sparse.hstack((tfidf_weighted, sparse.csr_matrix(mpnet_weighted)), format="csr")
    return np.concatenate((tfidf_weighted, mpnet_weighted), axis=1)


def summarize_clusters(items, labels):
    """Retorna atribuicoes e resumo de tamanho e frequencia por cluster."""
    result = items.copy()
    result["cluster"] = np.asarray(labels)
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary


def score_configuration(matrix, labels, sample_size, random_state):
    """Calcula cobertura, ruido e silhouette cosseno sem incluir ruido."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_positions = np.flatnonzero(~noise_mask)
    cluster_count = len(np.unique(labels[clustered_positions]))
    score = np.nan
    silhouette_count = min(sample_size, len(clustered_positions))
    if silhouette_count:
        rng = np.random.default_rng(random_state)
        sample_positions = np.sort(rng.choice(clustered_positions, size=silhouette_count, replace=False))
        sample_labels = labels[sample_positions]
        sample_cluster_count = len(np.unique(sample_labels))
        if 2 <= sample_cluster_count < silhouette_count:
            score = float(silhouette_score(matrix[sample_positions], sample_labels, metric="cosine"))
        else:
            silhouette_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_sample_size": silhouette_count,
        "silhouette_cosine_without_noise": score,
    }


def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Calcula uma projecao t-SNE 2D ou 3D reprodutivel sobre uma amostra."""
    point_count = features.shape[0]
    if point_count < 3:
        raise ValueError("O t-SNE requer pelo menos tres descricoes.")
    if n_components not in (2, 3):
        raise ValueError("n_components deve ser 2 ou 3.")
    sample_count = min(point_count, max_points)
    if sample_count < 3:
        raise ValueError("TSNE_MAX_POINTS deve ser pelo menos tres.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled_features = features[indices]
    if sampled_features.shape[1] > 1:
        component_count = min(TSNE_SVD_COMPONENTS, sampled_features.shape[1] - 1, sample_count - 1)
        reduced = TruncatedSVD(n_components=component_count, random_state=random_state).fit_transform(sampled_features)
    else:
        component_count = 1
        reduced = sampled_features.toarray() if sparse.issparse(sampled_features) else sampled_features
    effective_perplexity = min(float(perplexity), (sample_count - 1) / 3)
    coordinates = TSNE(
        n_components=n_components, perplexity=effective_perplexity,
        init="random" if n_components == 3 else "pca",
        learning_rate="auto", max_iter=max_iter, random_state=random_state,
    ).fit_transform(reduced)
    return indices, coordinates, component_count

def save_tsne_plot(coordinates, labels, title, output_path):
    """Salva uma visualizacao t-SNE 2D colorida pelas atribuicoes HDBSCAN."""
    labels = np.asarray(labels)
    figure, axis = plt.subplots(figsize=(12, 9))
    noise = labels == -1
    axis.scatter(coordinates[noise, 0], coordinates[noise, 1], s=10, c="#999999", alpha=0.35, label="Ruido (-1)")
    cluster_labels = np.unique(labels[~noise])
    if len(cluster_labels):
        colors = plt.get_cmap("turbo", max(len(cluster_labels), 1))
        for position, cluster in enumerate(cluster_labels):
            mask = labels == cluster
            axis.scatter(coordinates[mask, 0], coordinates[mask, 1], s=12, color=colors(position), alpha=0.75, label=f"Cluster {cluster}")
    axis.set(title=title, xlabel="t-SNE 1", ylabel="t-SNE 2")
    if len(cluster_labels) <= 15:
        axis.legend(loc="best", fontsize="small")
    figure.tight_layout()
    figure.savefig(output_path, dpi=160)
    plt.close(figure)



def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualizacao t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(point[0]), "y": float(point[1]), "z": float(point[2]), "label": int(label)}
        for point, label in zip(coordinates, labels)
    ]
    points_json = json.dumps(points, separators=(",", ":"), ensure_ascii=False).replace("<", "\\u003c")
    title = json.dumps(str(run_id), ensure_ascii=False).replace("<", "\\u003c")
    page = r"""<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>t-SNE 3D</title><style>
html,body{margin:0;height:100%;font:14px system-ui,sans-serif;color:#202124;background:#f7f8fa}main{display:flex;flex-direction:column;height:100%}header{padding:12px 18px;background:white;border-bottom:1px solid #ddd}h1{margin:0 0 4px;font-size:17px}p{margin:0;color:#555}canvas{display:block;width:100%;flex:1;min-height:300px;cursor:grab}canvas:active{cursor:grabbing}
</style></head><body><main><header><h1 id="title"></h1><p>Arraste para girar; use a roda do mouse para aproximar. Ruido em cinza.</p></header><canvas id="plot"></canvas></main>
<script>const points=__POINTS__,runTitle=__TITLE__,canvas=document.getElementById("plot"),ctx=canvas.getContext("2d");document.getElementById("title").textContent="t-SNE 3D: "+runTitle;
let yaw=.65,pitch=-.35,zoom=1,dragging=false,lastX=0,lastY=0;
const colors=["#3366cc","#dc3912","#ff9900","#109618","#990099","#0099c6","#dd4477","#66aa00","#b82e2e","#316395","#994499","#22aa99","#aaaa11","#6633cc","#e67300","#8b0707","#651067","#329262","#5574a6","#3b3eac"];
const bounds=["x","y","z"].map(k=>{const a=points.map(p=>p[k]),lo=Math.min(...a),hi=Math.max(...a);return{lo,span:Math.max(hi-lo,1e-9)}});
function resize(){const r=devicePixelRatio||1;canvas.width=Math.max(1,Math.floor(canvas.clientWidth*r));canvas.height=Math.max(1,Math.floor(canvas.clientHeight*r));ctx.setTransform(r,0,0,r,0,0);draw()}
function draw(){const w=canvas.clientWidth,h=canvas.clientHeight;ctx.clearRect(0,0,w,h);const scale=Math.min(w,h)*.7*zoom,cy=Math.cos(yaw),sy=Math.sin(yaw),cp=Math.cos(pitch),sp=Math.sin(pitch);const projected=points.map(p=>{const x=((p.x-bounds[0].lo)/bounds[0].span-.5)*2,y=((p.y-bounds[1].lo)/bounds[1].span-.5)*2,z=((p.z-bounds[2].lo)/bounds[2].span-.5)*2,rx=x*cy-z*sy,rz=x*sy+z*cy,ry=y*cp-rz*sp,d=y*sp+rz*cp;return{x:w/2+rx*scale/2,y:h/2-ry*scale/2,d,label:p.label}}).sort((a,b)=>a.d-b.d);for(const p of projected){ctx.globalAlpha=p.label === -1 ? 0.24 : 0.78;ctx.fillStyle=p.label===-1?"#999999":colors[Math.abs(p.label)%colors.length];ctx.beginPath();ctx.arc(p.x,p.y,p.label === -1 ? 2 : 3,0,Math.PI*2);ctx.fill()}ctx.globalAlpha=1}
canvas.addEventListener("pointerdown",e=>{dragging=true;lastX=e.clientX;lastY=e.clientY;canvas.setPointerCapture(e.pointerId)});canvas.addEventListener("pointerup",()=>dragging=false);canvas.addEventListener("pointermove",e=>{if(!dragging)return;yaw+=(e.clientX-lastX)*.008;pitch=Math.max(-1.5,Math.min(1.5,pitch+(e.clientY-lastY)*.008));lastX=e.clientX;lastY=e.clientY;draw()});canvas.addEventListener("wheel",e=>{e.preventDefault();zoom=Math.max(.3,Math.min(5,zoom*(e.deltaY<0?1.1:.9)));draw()},{passive:false});window.addEventListener("resize",resize);resize();</script></body></html>"""
    page = page.replace("__POINTS__", points_json).replace("__TITLE__", title)
    output_path.write_text(page, encoding="utf-8")


## 3. Carregar e preparar os dados

Origem: o Parquet do periodo encontrado em `data/processed/2025_NCMs_filtrados-sem-duplicatas/`. Descricoes duplicadas sao agregadas; a amostra e fixa e ordenada para manter o alinhamento entre TF-IDF e MPNet.


In [3]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes unicas disponiveis: %d", len(base_items))
sampled_items = base_items
if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
    sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=SAMPLE_RANDOM_STATE)
sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
sampled_items.insert(0, "descricao_id", np.arange(len(sampled_items), dtype=np.int64))
if sampled_items.empty:
    raise ValueError("Nao ha descricoes validas para o experimento.")
display(sampled_items.head())
logger.info("Descricoes na amostra: %d", len(sampled_items))


2026-10-18 10:46:06,289 INFO Descricoes unicas disponiveis: 149149


,descricao_id,descricao_normalizada,frequencia
0,0,00007 detergente composicao tensoativo anionico biodegradavel ph 6 5 7 5 c aroma neutro aspecto fisico l,1
1,1,00054 bolo alimenticio sabor maracuja tipo torta confeitada peso 2 kg caracteristicas adicionais com 30 cm d,2
2,2,0009 saco plastico ksf 1050x2500,1
3,3,00099 salgados diversos tipo rissole tamanho pequeno aplicacao alimentacao sabor presunto e queijo,1
4,4,001 126 pilha tam peq aa 1 5v alcalina,1


2026-10-18 10:46:06,342 INFO Descricoes na amostra: 10000


## 4. Gerar embeddings MPNet

MPNet e calculado uma vez e reutilizado nas configuracoes TF-IDF. Os embeddings sao normalizados na funcao de combinacao, conforme o notebook 52.


In [4]:
texts = sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist()
row_ids = sampled_items["descricao_id"].to_numpy()
logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)
mpnet_matrix = encoder.encode(
    texts, batch_size=ENCODE_BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=False,
).astype(np.float32, copy=False)
if mpnet_matrix.shape[0] != len(sampled_items):
    raise ValueError("MPNet nao preservou uma linha por descricao.")
logger.info("Embeddings MPNet: %d x %d", *mpnet_matrix.shape)


2026-10-18 10:46:06,407 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-10-18 10:46:06,410 INFO No device provided, using cpu
2026-10-18 10:46:06,652 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-18 10:46:06,652 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-18 10:46:06,687 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-10-18 10:46:06,837 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/pa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-10-18 10:46:09,808 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-18 10:46:09,967 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-18 10:46:10,107 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-18 10:46:12,880 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-18 10:46:13,053 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-18 10:46:13,077 INFO HTTP Request: HEAD https:

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-10-18 10:53:01,826 INFO Embeddings MPNet: 10000 x 768


## 5. Ajustar TF-IDF + HDBSCAN

A grade cruza cinco analyzers com `min_cluster_size` 5/10/20 e `min_samples` padrao/5/10. Silhouette cosseno exclui ruido e usa ate 1.000 descricoes classificadas. Cada descricao unica tem o mesmo peso no HDBSCAN; frequencia e preservada nos relaterios.

A matriz hibrida e CSR, mantendo TF-IDF esparso e convertendo MPNet para CSR. O bloco MPNet ainda pode consumir memoria consideravel.


In [5]:
records = []
labels_by_run = {}
items_by_run = {}
hybrid_matrices = {}
vectorizers = {}

for config_name, (analyzer, ngram_range) in ANALYZER_CONFIGS.items():
    logger.info("Ajustando TF-IDF: %s (%s, %s)", config_name, analyzer, ngram_range)
    vectorizer = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngram_range, min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES, sublinear_tf=True, norm=None,
    )
    tfidf_matrix = vectorizer.fit_transform(texts)
    if tfidf_matrix.shape[0] != len(sampled_items):
        raise ValueError(f"TF-IDF ({config_name}) nao preservou uma linha por descricao.")
    hybrid_matrix = combine_representations(tfidf_matrix, mpnet_matrix, ALPHA, row_ids, row_ids.copy())
    hybrid_matrices[config_name] = hybrid_matrix
    vectorizers[config_name] = vectorizer
    logger.info("%s: TF-IDF=%d features; hibrida=%d dimensoes; %d valores", config_name, tfidf_matrix.shape[1], hybrid_matrix.shape[1], hybrid_matrix.nnz)

    for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            min_samples_label = "default" if min_samples is None else str(min_samples)
            run_id = (f"seed{SAMPLE_RANDOM_STATE}_{config_name}_alpha{ALPHA:g}"
                      f"_min_cluster_size{min_cluster_size}_min_samples{min_samples_label}_{CLUSTER_SELECTION_METHOD}")
            logger.info("Ajustando HDBSCAN %s", run_id)
            model = HDBSCAN(
                min_cluster_size=min_cluster_size, min_samples=min_samples,
                metric="cosine", algorithm="brute",
                cluster_selection_method=CLUSTER_SELECTION_METHOD,
                n_jobs=HDBSCAN_N_JOBS, copy=True,
            )
            labels = model.fit_predict(hybrid_matrix)
            metrics = score_configuration(hybrid_matrix, labels, SILHOUETTE_SAMPLE_SIZE, TSNE_RANDOM_STATE + SAMPLE_RANDOM_STATE)
            records.append({
                "run_id": run_id, "sample_seed": SAMPLE_RANDOM_STATE,
                "representation": "tfidf_mpnet_hybrid", "tfidf_config": config_name,
                "analyzer": analyzer, "ngram_range": str(ngram_range), "alpha": ALPHA,
                "tfidf_features": tfidf_matrix.shape[1], "mpnet_dimensions": mpnet_matrix.shape[1],
                "hybrid_dimensions": hybrid_matrix.shape[1], "model_name": MODEL_NAME,
                "min_cluster_size": min_cluster_size, "min_samples": min_samples,
                "cluster_selection_method": CLUSTER_SELECTION_METHOD,
                "frequency_weighted": False, **metrics,
            })
            labels_by_run[run_id] = labels
            items_by_run[run_id] = sampled_items

results = pd.DataFrame(records)
logger.info("Configuracoes HDBSCAN avaliadas: %d", len(results))


2026-10-18 10:53:01,931 INFO Ajustando TF-IDF: char_wb_3_5 (char_wb, (3, 5))
2026-10-18 10:53:04,231 INFO char_wb_3_5: TF-IDF=15853 features; hibrida=16621 dimensoes; 8331569 valores
2026-10-18 10:53:04,233 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size5_min_samplesdefault_eom
2026-10-18 10:55:48,822 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size5_min_samples5_eom
2026-10-18 10:58:52,388 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size5_min_samples10_eom
2026-10-18 11:01:30,545 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samplesdefault_eom
2026-10-18 11:05:09,915 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samples5_eom
2026-10-18 11:06:18,667 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samples10_eom
2026-10-18 11:08:35,280 INFO Ajustando HDBSCAN seed42_char_wb_3_5_alpha0.2_min_cluster_size20_min_samplesdefault_eom
2026-10-18 11:11:14,008

## 6. Filtrar e comparar configuracoes

O score combina silhouette reescalada para 0-1 e cobertura (`1 - noise_fraction`) com pesos iguais, como nos notebooks 74 e 75. O ranking aplica o limite de clusters e, por padrao, exige silhouette valida maior que a fracao de ruido. Esses filtros afetam apenas a ordenacao e selecao final, nao os ajustes. Se nenhum resultado passar, `SHOW_FULL_BASE_IF_RANKING_EMPTY` controla se a tabela completa sera exibida como fallback.


In [6]:
if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deve ser maior que zero.")
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (SILHOUETTE_WEIGHT * results["silhouette_quality"] + NOISE_WEIGHT * results["noise_quality"]) / weight_total
ranking_mask = results["cluster_count_excluding_noise"].le(MAX_CLUSTERS_FOR_RANKING)
if RANKING_FILTER_SILHOUETTE_ABOVE_NOISE:
    ranking_mask &= (
        results["silhouette_cosine_without_noise"].notna()
        & results["silhouette_cosine_without_noise"].gt(results["noise_fraction"])
    )
ranking = (
    results.loc[ranking_mask]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
RANKING_USED_FULL_BASE = False
if ranking.empty and SHOW_FULL_BASE_IF_RANKING_EMPTY:
    logger.warning(
        "Os filtros do ranking nao selecionaram resultados; exibindo a base completa de configuracoes."
    )
    ranking = (
        results.sort_values(
            ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
            ascending=[False, False, True], na_position="last",
        )
        .reset_index(drop=True)
    )
    RANKING_USED_FULL_BASE = True
elif ranking.empty:
    raise ValueError(
        "Nenhum resultado atende aos filtros do ranking. Ajuste os parametros, "
        "desative os filtros ou habilite SHOW_FULL_BASE_IF_RANKING_EMPTY."
    )
ranking.insert(0, "rank", range(1, len(ranking) + 1))

if SELECTED_RUN_ID is None:
    SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
if SELECTED_RUN_ID not in labels_by_run:
    raise ValueError(f"SELECTED_RUN_ID nao encontrado: {SELECTED_RUN_ID}")
display(ranking)


2026-10-18 13:04:30,656 WARNING Os filtros do ranking nao selecionaram resultados; exibindo a base completa de configuracoes.


,rank,run_id,sample_seed,representation,tfidf_config,analyzer,ngram_range,alpha,tfidf_features,mpnet_dimensions,...,cluster_selection_method,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samplesdefault_eom,42,tfidf_mpnet_hybrid,char_wb_3_5,char_wb,"(3, 5)",0.2,15853,768,...,eom,False,2,1135,0.1135,8865,0.097649,0.548824,0.8865,0.689523
1,2,seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samples10_eom,42,tfidf_mpnet_hybrid,char_wb_3_5,char_wb,"(3, 5)",0.2,15853,768,...,eom,False,2,1135,0.1135,8865,0.097649,0.548824,0.8865,0.689523
2,3,seed42_word_1_3_alpha0.2_min_cluster_size10_min_samplesdefault_eom,42,tfidf_mpnet_hybrid,word_1_3,word,"(1, 3)",0.2,3000,768,...,eom,False,2,1267,0.1267,8733,0.104665,0.552332,0.8733,0.686069
3,4,seed42_word_1_3_alpha0.2_min_cluster_size10_min_samples10_eom,42,tfidf_mpnet_hybrid,word_1_3,word,"(1, 3)",0.2,3000,768,...,eom,False,2,1267,0.1267,8733,0.104665,0.552332,0.8733,0.686069
4,5,seed42_word_1_3_alpha0.2_min_cluster_size20_min_samples10_eom,42,tfidf_mpnet_hybrid,word_1_3,word,"(1, 3)",0.2,3000,768,...,eom,False,2,3721,0.3721,6279,0.051440,0.525720,0.6279,0.568295
5,6,seed42_word_1_5_alpha0.2_min_cluster_size20_min_samples10_eom,42,tfidf_mpnet_hybrid,word_1_5,word,"(1, 5)",0.2,3070,768,...,eom,False,2,3721,0.3721,6279,0.051434,0.525717,0.6279,0.568293
6,7,seed42_word_2_5_alpha0.2_min_cluster_size20_min_samples5_eom,42,tfidf_mpnet_hybrid,word_2_5,word,"(2, 5)",0.2,1239,768,...,eom,False,6,4078,0.4078,5922,0.010405,0.505203,0.5922,0.541451
7,8,seed42_char_wb_5_10_alpha0.2_min_cluster_size5_min_samplesdefault_eom,42,tfidf_mpnet_hybrid,char_wb_5_10,char_wb,"(5, 10)",0.2,18957,768,...,eom,False,214,6602,0.6602,3398,0.358150,0.679075,0.3398,0.537710
8,9,seed42_char_wb_5_10_alpha0.2_min_cluster_size5_min_samples5_eom,42,tfidf_mpnet_hybrid,char_wb_5_10,char_wb,"(5, 10)",0.2,18957,768,...,eom,False,214,6602,0.6602,3398,0.358150,0.679075,0.3398,0.537710
9,10,seed42_word_2_5_alpha0.2_min_cluster_size5_min_samplesdefault_eom,42,tfidf_mpnet_hybrid,word_2_5,word,"(2, 5)",0.2,1239,768,...,eom,False,221,7514,0.7514,2486,0.463615,0.731808,0.2486,0.530471


## 7. Salvar metricas e todos os t-SNE

Esta celula salva metricas e ranking, coordenadas 2D por configuracao TF-IDF e os graficos 2D/3D para todos os runs. Ela nao salva assignments nem modelo. Use a celula seguinte para salvar esses artifacts de um `ARTIFACT_RUN_ID` especifico, sem recalcular os t-SNE.


In [7]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False, encoding="utf-8-sig")
ranking.to_csv(output_dir / "ranking.csv", index=False, encoding="utf-8-sig")

# Calcula as projecoes para cada matriz e reutiliza cada uma nos nove runs HDBSCAN.
tsne_by_config = {}
tsne_3d_by_config = {}
for config_name, matrix in hybrid_matrices.items():
    logger.info("Calculando t-SNE 2D para %s", config_name)
    indices_2d, coordinates_2d, dimensions_2d = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE + SAMPLE_RANDOM_STATE, n_components=2,
    )
    logger.info("Calculando t-SNE 3D para %s", config_name)
    indices_3d, coordinates_3d, dimensions_3d = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE + SAMPLE_RANDOM_STATE, n_components=TSNE_3D_N_COMPONENTS,
    )
    if not np.array_equal(indices_2d, indices_3d):
        raise ValueError("As projecoes 2D e 3D precisam usar a mesma amostra.")
    tsne_by_config[config_name] = {
        "indices": indices_2d, "coordinates": coordinates_2d,
        "dimensions": dimensions_2d,
    }
    tsne_3d_by_config[config_name] = {
        "indices": indices_3d, "coordinates": coordinates_3d,
        "dimensions": dimensions_3d,
    }

coordinates_dir = output_dir / "tsne_coordinates"
plots_dir = output_dir / "tsne_plots"
html_3d_dir = output_dir / "tsne_3d_html"
coordinates_dir.mkdir()
plots_dir.mkdir()
html_3d_dir.mkdir()
tsne_artifacts = {}
for config_name, projection in tsne_by_config.items():
    indices = projection["indices"]
    frame = sampled_items.iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    frame["tsne_1"] = projection["coordinates"][:, 0]
    frame["tsne_2"] = projection["coordinates"][:, 1]
    coordinate_path = coordinates_dir / f"{config_name}_coordinates.csv"
    frame.to_csv(coordinate_path, index=False, encoding="utf-8-sig")
    tsne_artifacts[f"coordinates:{config_name}"] = str(
        coordinate_path.relative_to(output_dir)
    )

for run_id, labels in labels_by_run.items():
    config_name = results.loc[results["run_id"].eq(run_id), "tfidf_config"].iloc[0]
    projection = tsne_by_config[config_name]
    projection_3d = tsne_3d_by_config[config_name]
    labels_2d = np.asarray(labels)[projection["indices"]]
    labels_3d = np.asarray(labels)[projection_3d["indices"]]
    plot_path = plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], labels_2d, run_id, plot_path)
    html_path = html_3d_dir / f"{run_id}.html"
    save_tsne_3d_html(projection_3d["coordinates"], labels_3d, run_id, html_path)
    tsne_artifacts[f"plot_2d:{run_id}"] = str(plot_path.relative_to(output_dir))
    tsne_artifacts[f"plot_3d:{run_id}"] = str(html_path.relative_to(output_dir))

run_metadata = {
    "dataset_name": DATASET_NAME,
    "dataset_path": str(DATASET_PATH.relative_to(PROJECT_ROOT)),
    "period": PERIOD_LABEL,
    "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seed": SAMPLE_RANDOM_STATE,
    "representation": "tfidf_mpnet_hybrid",
    "alpha": ALPHA,
    "model_name": MODEL_NAME,
    "analyzer_configs": {
        name: {"analyzer": analyzer, "ngram_range": list(ngram_range)}
        for name, (analyzer, ngram_range) in ANALYZER_CONFIGS.items()
    },
    "tfidf_min_df": TFIDF_MIN_DF,
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "min_cluster_size_values": MIN_CLUSTER_SIZE_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "ranking_filter_silhouette_above_noise": RANKING_FILTER_SILHOUETTE_ABOVE_NOISE,
    "show_full_base_if_ranking_empty": SHOW_FULL_BASE_IF_RANKING_EMPTY,
    "ranking_used_full_base": RANKING_USED_FULL_BASE,
    "selected_run_id": SELECTED_RUN_ID,
    "output_dir": str(output_dir.relative_to(PROJECT_ROOT)),
    "tsne": {
        "max_points": TSNE_MAX_POINTS,
        "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER,
        "random_state": TSNE_RANDOM_STATE + SAMPLE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS,
        "three_dimensional_components": TSNE_3D_N_COMPONENTS,
        "artifacts": tsne_artifacts,
    },
    "frequency_weighted": False,
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Metricas e todos os graficos t-SNE salvos em %s", output_dir.relative_to(PROJECT_ROOT))


2026-10-18 13:04:30,935 INFO Calculando t-SNE 2D para char_wb_3_5
2026-10-18 13:05:04,862 INFO Calculando t-SNE 3D para char_wb_3_5
2026-10-18 13:06:36,491 INFO Calculando t-SNE 2D para char_wb_5_10
2026-10-18 13:07:13,079 INFO Calculando t-SNE 3D para char_wb_5_10
2026-10-18 13:08:49,016 INFO Calculando t-SNE 2D para word_1_3
2026-10-18 13:09:20,497 INFO Calculando t-SNE 3D para word_1_3
2026-10-18 13:10:37,595 INFO Calculando t-SNE 2D para word_1_5
2026-10-18 13:11:04,919 INFO Calculando t-SNE 3D para word_1_5
2026-10-18 13:12:22,427 INFO Calculando t-SNE 2D para word_2_5
2026-10-18 13:12:50,727 INFO Calculando t-SNE 3D para word_2_5
2026-10-18 13:14:39,958 INFO Metricas e todos os graficos t-SNE salvos em reports\tfidf_mpnet_hdbscan\202501-202512_20261018T160430Z860319


## 8. Salvar assignments e modelo de um run especifico

Informe o `ARTIFACT_RUN_ID` e execute somente a celula de codigo abaixo. Ela salva o CSV de assignments e o modelo HDBSCAN junto do vetorizador TF-IDF, encoder MPNet e configuracao. Esta celula nao calcula nem salva t-SNE.


In [8]:
ARTIFACT_RUN_ID = "seed42_word_2_5_alpha0.5_min_cluster_size10_min_samples5_eom"

if ARTIFACT_RUN_ID not in labels_by_run:
    raise ValueError(
        f"ARTIFACT_RUN_ID nao encontrado. Runs disponiveis: {list(labels_by_run)}"
    )

artifact_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
artifact_folder = f"{PERIOD_LABEL}_{artifact_timestamp}_{ARTIFACT_RUN_ID}"
assignment_output_dir = OUTPUT_ROOT / f"assignments_{artifact_folder}"
model_output_dir = MODEL_ROOT / artifact_folder
assignment_output_dir.mkdir(parents=True, exist_ok=False)
model_output_dir.mkdir(parents=True, exist_ok=False)

artifact_items = items_by_run[ARTIFACT_RUN_ID]
artifact_labels = labels_by_run[ARTIFACT_RUN_ID]
assignments, artifact_cluster_summary = summarize_clusters(artifact_items, artifact_labels)
assignment_path = assignment_output_dir / f"assignments_{ARTIFACT_RUN_ID}.csv"
assignments.to_csv(assignment_path, index=False, encoding="utf-8-sig")

artifact_run = results.set_index("run_id").loc[ARTIFACT_RUN_ID]
config_name = artifact_run["tfidf_config"]
min_samples = None if pd.isna(artifact_run["min_samples"]) else int(artifact_run["min_samples"])
saved_model = HDBSCAN(
    min_cluster_size=int(artifact_run["min_cluster_size"]),
    min_samples=min_samples,
    metric="cosine",
    algorithm="brute",
    cluster_selection_method=artifact_run["cluster_selection_method"],
    n_jobs=HDBSCAN_N_JOBS,
    copy=True,
)
saved_model.fit(hybrid_matrices[config_name])
if not np.array_equal(saved_model.labels_, artifact_labels):
    raise RuntimeError(
        "Os labels do modelo refeito nao coincidem com os assignments do ARTIFACT_RUN_ID."
    )

encoder.save(str(model_output_dir / "sentence_transformer"))
pd.to_pickle(saved_model, model_output_dir / "hdbscan_model.pkl")
pd.to_pickle(vectorizers[config_name], model_output_dir / "tfidf_vectorizer.pkl")
assignments.to_csv(
    model_output_dir / "training_assignments.csv", index=False, encoding="utf-8-sig"
)
model_config = {
    "run_id": ARTIFACT_RUN_ID,
    "period": PERIOD_LABEL,
    "dataset_name": DATASET_NAME,
    "sample_seed": SAMPLE_RANDOM_STATE,
    "representation": "tfidf_mpnet_hybrid",
    "tfidf": {
        "config_name": config_name,
        "analyzer": artifact_run["analyzer"],
        "ngram_range": artifact_run["ngram_range"],
        "min_df": TFIDF_MIN_DF,
        "max_features": TFIDF_MAX_FEATURES,
        "sublinear_tf": True,
        "norm": None,
        "vectorizer_artifact": "tfidf_vectorizer.pkl",
    },
    "encoder": {
        "model_name": MODEL_NAME,
        "artifact": "sentence_transformer/",
        "encode_batch_size": ENCODE_BATCH_SIZE,
        "normalize_embeddings_before_combination": True,
        "embedding_dimension": int(mpnet_matrix.shape[1]),
    },
    "combination": {
        "alpha": ALPHA,
        "normalization": "l2 per block",
        "weights": {
            "tfidf": float(np.sqrt(ALPHA)),
            "mpnet": float(np.sqrt(1.0 - ALPHA)),
        },
    },
    "hdbscan": {
        "min_cluster_size": int(artifact_run["min_cluster_size"]),
        "min_samples": min_samples,
        "metric": "cosine",
        "algorithm": "brute",
        "cluster_selection_method": artifact_run["cluster_selection_method"],
        "artifact": "hdbscan_model.pkl",
    },
    "frequency_weighted": False,
    "assignments_csv": str(assignment_path.relative_to(PROJECT_ROOT)),
    "training_assignments": "training_assignments.csv",
    "prediction_note": (
        "HDBSCAN foi ajustado sem prediction_data; o modelo nao atribui diretamente "
        "novas descricoes a clusters."
    ),
}
(model_output_dir / "config.json").write_text(
    json.dumps(model_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Assignments salvos em %s", assignment_path.relative_to(PROJECT_ROOT))
logger.info("Modelo salvo em %s", model_output_dir.relative_to(PROJECT_ROOT))


ValueError: ARTIFACT_RUN_ID nao encontrado. Runs disponiveis: ['seed42_char_wb_3_5_alpha0.2_min_cluster_size5_min_samplesdefault_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size5_min_samples5_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size5_min_samples10_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samplesdefault_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samples5_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size10_min_samples10_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size20_min_samplesdefault_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size20_min_samples5_eom', 'seed42_char_wb_3_5_alpha0.2_min_cluster_size20_min_samples10_eom', 'seed42_char_wb_5_10_alpha0.2_min_cluster_size5_min_samplesdefault_eom', 'seed42_char_wb_5_10_alpha0.2_min_cluster_size5_min_samples5_eom', 'seed42_char_wb_5_10_alpha0.2_min_cluster_size5_min_samples10_eom', 'seed42_char_wb_5_10_alpha0.2_min_cluster_size10_min_samplesdefault_eom', 'seed42_char_wb_5_10_alpha0.2_min_

## 9. Interpretacao e limites

- O ranking filtra por quantidade maxima de clusters e, por padrao, exige silhouette nao nula e maior que a fracao de ruido. Desative `RANKING_FILTER_SILHOUETTE_ABOVE_NOISE` para remover essa segunda regra.
- Cada descricao unica tem o mesmo peso no HDBSCAN; frequencia aparece nos relatorios.
- A silhouette exclui ruido e usa ate 1.000 itens. Compare-a com cobertura e quantidade de clusters.
- t-SNE e uma projecao exploratoria; o HTML 3D permite girar e aproximar a nuvem de pontos.
- HDBSCAN e salvo sem mecanismo de predicao para novos pontos; o encoder e o vetorizador ficam disponiveis para reproduzir a representacao de treino.
- Os clusters sao exploratorios, nao categorias fiscais validadas.
